# 04 — Compute ACE Descriptors

Computes Atomic Cluster Expansion (ACE) descriptors for all training structures.

## Prerequisites / Input files
- `Fe-Mo/Atomsobjects/Fe-Mo-POSCAR-initial-rescaled-AtomsObjects.pkl`
- `python-ace` package (public, included in `environment_public.yaml`)

## Outputs
- `Fe-Mo/Descriptors/Fe-Mo-*-ACE-CNAV.csv`

## Notes
> ⚠️ ACE descriptor computation may require a separate Python environment. Test with `conda activate datasets_ml` before running.



#  Calculation of features from available libraries

In [ ]:
from Tools.DatasetTools.Commoms import *

In [ ]:
# Skip descriptor computation if output files already exist
_ace_output = os.path.join('Fe-Mo', 'Descriptors', 'Fe-Mo-ACE-CNAV.csv')
_ace_descriptors_exist = os.path.exists(_ace_output)
if _ace_descriptors_exist:
    print('ACE descriptors found — skipping recomputation.')


In [ ]:
plt.rc('text', usetex=False)
plt.rc('font', family='serif', size=24)

In [ ]:
dataset = 'Fe-Mo'# 'Fe-Mo'  # 'Cr-Co-W' # 
components = dataset.split('-')
system=dataset.replace('-','')
try:
    from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.Featurizer import Featurizer
    _HAS_BOPFOX = True
except (ImportError, Exception):
    _HAS_BOPFOX = False

import Tools.DatasetTools.GeneralFeaturizer as gf

BS = pd.read_pickle(os.path.join(dataset, 'FullyCuratedParsedBriefSummary.pkl'))

In [ ]:
BS.describe()

In [ ]:
if not _ace_descriptors_exist:
    AtomsObjects = pd.read_pickle(os.path.join(dataset, 'Atomsobjects', f'{dataset}-POSCAR-initial-rescaled-AtomsObjects.pkl')).dropna()
    
    PymatgenStructures = pd.read_pickle(os.path.join(dataset, 'Atomsobjects', f'{dataset}-POSCAR-initial-rescaled-PymatgenStructures.pkl')).dropna()
    SublatticeTags = pd.read_pickle(os.path.join(dataset,'Atomsobjects', 'SUBLATICETAGS_POSCAR.initial.pkl'))
    SublatticeSorters = pd.read_pickle(os.path.join(dataset,'Atomsobjects', 'SORTERS_POSCAR.initial.pkl'))
    SublatticeSorters.index = SublatticeSorters.index.astype(str).str.strip()
    SublatticeTags.index = SublatticeSorters.index.astype(str).str.strip()
    
    BS.dropna(inplace=True)
    
    import numpy as np
    
    descriptorslocation = os.path.join(dataset, 'Descriptors')

In [ ]:
if not _ace_descriptors_exist:
    from Tools.DatasetTools.GeneralFeaturizer import cn_persite

# Prepare Extra features

In [ ]:
if not _ace_descriptors_exist:
    from importlib.machinery import SourceFileLoader

In [ ]:
if not _ace_descriptors_exist:
    from sklearn.preprocessing import  OneHotEncoder, LabelEncoder
    encoder = LabelEncoder()

In [ ]:
if not _ace_descriptors_exist:
    from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.Featurizer import Featurizer

In [ ]:
if not _ace_descriptors_exist:
    Features = Featurizer(BS)

In [ ]:
if not _ace_descriptors_exist:
    DatasetCompositionFeatures = Features.get_fractions_by_components()

In [ ]:
if not _ace_descriptors_exist:
    #DatasetFeatures = pd.concat((DatasetCompositionFeatures, DatasetMagneticFeature, StructureNameFeature), axis=1)
    pass

##  Magnetism and structure

In [ ]:
if not _ace_descriptors_exist:
    StructureNameFeature = BS.Phase
    StructureNameFeature.name='Structure'
    encoder.fit(StructureNameFeature)
    DatasetStructureFeature = pd.Series(encoder.transform(StructureNameFeature), name='Structure', index = StructureNameFeature.index)

In [ ]:
if not _ace_descriptors_exist:
    MagneticFeature = Features.MagFeature
    MagneticFeature.name = 'Mag'
    encoder.fit(MagneticFeature)
    DatasetMagneticFeature = pd.Series(encoder.transform(MagneticFeature), name='Mag', index = StructureNameFeature.index)

In [ ]:
if not _ace_descriptors_exist:
    DatasetFeatures = pd.concat([DatasetMagneticFeature, DatasetStructureFeature, DatasetCompositionFeatures, BS.num_atoms], axis = 1)

## Coordination Polyhedra feature

The first feature that we would like to have is the count of each CP in each sample. for that we construct a vector in the following way:

$$ N_{CN}^i = \#^i CP $$

Next feature we want is the composition in each CP. for this we choose to represent the elment numerically by their atomic numbers, and the CP-resolved composition becomes the average atomc numbers,

$$ Z_{CP} ^i = \dfrac{1}{n_{at}^i} \sum_{at \in CP} Z_{at} $$

In [ ]:
if not _ace_descriptors_exist:
    BS.index

In [ ]:
if not _ace_descriptors_exist:
    AtomsObjects.index

In [ ]:
if not _ace_descriptors_exist:
    intersection = AtomsObjects.index.intersection(BS.index)

In [ ]:
if not _ace_descriptors_exist:
    AtomsObjects = AtomsObjects.loc[intersection]

In [ ]:
if not _ace_descriptors_exist:
    BS = BS.loc[intersection]

In [ ]:
if not _ace_descriptors_exist:
    SortingFeatures = gf.sorting_feature(AtomsObjects, SublatticeSorters, SublatticeTags)
    SortingFeatures.sorters = gf.correct_sortings_fromphases(AtomsObjects, BS.Phase, SortingFeatures.sorters)
    SortingFeatures.sublatticetags = gf.correct_occupation_fromphases(BS.Phase, SortingFeatures.sublatticetags, AtomsObjects.atoms)
    sampleinspecial = BS.Phase.map(lambda p: p in gf.specialphases)
    empty = SortingFeatures.sublatticetags.map(lambda sublat: '' in sublat)
    SortingFeatures.sublatticetags[empty] = ['A']
    wrong = SortingFeatures.sublatticetags.map(lambda sublat: 'A' not in sublat) 
    fixable = SortingFeatures.sublatticetags.loc[wrong].map(type) == np.ndarray #.map(np.unique)
    CNList = gf.get_sitecn(BS.Phase, AtomsObjects.atoms, SortingFeatures.sorters)

In [ ]:
if not _ace_descriptors_exist:
    SortingFeatures.sorters = SortingFeatures.sorters.loc[intersection]

## Position Features

In [ ]:
if not _ace_descriptors_exist:
    elements = np.unique(BS.filter(regex='^atom_').values.ravel())
    ABOCC = pd.concat([BS.filter(regex='atom_'), Features.occupation], axis = 1)
    ABOCC.rename(columns={ABOCC.columns[-1]: 'sublatticetags'}, inplace=True)

In [ ]:
if not _ace_descriptors_exist:
    ABOCC=ABOCC.loc[intersection]

In [ ]:
if not _ace_descriptors_exist:
    Positions = {}
    for index, item in ABOCC.iterrows():
        if item['sublatticetags'] == '':
            thisposition = {index: [item[f'atom_A']]*len(np.unique(gf.cn_dict[BS.Phase[index]]))}
        else:
            thisposition = {index: [item[f'atom_{occ}'] for occ in item['sublatticetags'] ]}
        Positions.update(thisposition)
        

In [ ]:
if not _ace_descriptors_exist:
    Positions = pd.DataFrame.from_dict(Positions, orient='index')
    Positions[Positions.isnull()] = 0
    for i, element in enumerate(elements):
        Positions[Positions==element] = i
    Positions.columns = [f'Pos_{col+1}' for col in Positions.columns]
    #Positions[Positions.Pos_1.map(type) == str] = np.nan

## Averages over Coordination polyhedra

### Number of each CP in each structure

In [ ]:
if not _ace_descriptors_exist:
    CN = gf.featurize_series(CNList, CNList, normalization='NCP', return0 = False)
    newcolumns = ['N'+col for col in CN.columns]
    CN.columns = newcolumns

### Composition and volume of the CP

In [ ]:
if not _ace_descriptors_exist:
    from mendeleev import element

In [ ]:
if not _ace_descriptors_exist:
    AtomicNumbers=AtomsObjects.atoms.map(lambda a: a.numbers)
    AtomicNumbers.name = 'AtomicNumbers'
    symbols = dataset.split('-')
    volums = {symb: element(symb).atomic_volume for symb in symbols}

In [ ]:
if not _ace_descriptors_exist:
    AtomicVolumes = AtomsObjects.atoms.map(lambda a: np.array([volums[at] for at in a.get_chemical_symbols()]))

In [ ]:
if not _ace_descriptors_exist:
    CPVol = gf.featurize_series(AtomicVolumes, CNList, return0=False, normalization='NCP')
    newcolumns = ['V'+col for col in CPVol.columns]
    CPVol.columns =  newcolumns

In [ ]:
if not _ace_descriptors_exist:
    CPComp = gf.featurize_series(AtomicNumbers, CNList, return0=False, normalization='NCP')
    newcolumns = ['Z'+col for col in CPComp.columns]
    CPComp.columns = newcolumns

## Compile all the descriptors

In [ ]:
if not _ace_descriptors_exist:
    DatasetFeatures = pd.concat([DatasetStructureFeature, DatasetMagneticFeature, DatasetCompositionFeatures, CN, CPVol, CPComp, BS.num_atoms, Positions], axis=1)
    datasetfeatureslocation = os.path.join(dataset, 'Descriptors','DatasetFeatures.pkl')
    CNListlocation = os.path.join(dataset, 'Descriptors', 'CNList.pkl')
    DatasetFeatures.to_pickle(datasetfeatureslocation)
    CNList.to_pickle(CNListlocation)

In [ ]:
if not _ace_descriptors_exist:
    BS['atoms_objects'] = PymatgenStructures

# ACE Features 

In [ ]:
if not _ace_descriptors_exist:
    from ase.atoms import Atoms

In [ ]:
if not _ace_descriptors_exist:
    def reset_symbols(a: Atoms, newsym : str = 'W'):
        newa = a.copy()
        natoms = newa.get_global_number_of_atoms()
        newsymbols = [newsym]*natoms
        newa.set_chemical_symbols(newsymbols)
        return newa

In [ ]:
if not _ace_descriptors_exist:
    from importlib.machinery import SourceFileLoader

In [ ]:
if not _ace_descriptors_exist:
    from Tools.DatasetTools.ACEDescriptors import MyPyACECalculator 
    from Tools.DatasetTools.ACEDescriptors import default_options_dict as default_options_dict
    from pyace import ACEBBasisSet, PyACECalculator

In [ ]:
if not _ace_descriptors_exist:
    AceConfig = copy.copy(default_options_dict)

In [ ]:
if not _ace_descriptors_exist:
    AceConfig['elements'] = dataset.split('-')

In [ ]:
if not _ace_descriptors_exist:
    ACEer = MyPyACECalculator(components = components, multispace_basis_config = AceConfig)

In [ ]:
if not _ace_descriptors_exist:
    ACEer.bbasis_configuration.total_number_of_functions

In [ ]:
if not _ace_descriptors_exist:
    ACEFEATURES = AtomsObjects['atoms'].map(ACEer.get_ace_projections)
    ACEFEATURES.name = 'ace_projections'

In [ ]:
if not _ace_descriptors_exist:
    expand_ace = gf.array_expansions(ACEFEATURES.to_frame(),['ace_projections']) 

In [ ]:
if not _ace_descriptors_exist:
    CNAV_ACE = gf.featurize_dataframe(expand_ace, CNList).astype(np.single)

In [ ]:
if not _ace_descriptors_exist:
    CNAV_ACE

In [ ]:
if not _ace_descriptors_exist:
    selection = CNAV_ACE[Features.StrucNames.loc[intersection] == 'bcc'].filter(regex='_0$')

In [ ]:
if not _ace_descriptors_exist:
    selection

In [ ]:
if not _ace_descriptors_exist:
    fig, ax = plt.subplots()
    ax.plot(selection.iloc[0].values, 'o')
    ax.plot(selection.iloc[1].values, 'o')
    ax.plot(selection.iloc[2].values, 'o')
    ax.plot(selection.iloc[3].values, 'o')
    ax.set_yscale('log')
    ax.set_ylabel('ace projection')
    ax.set_xlabel('lexycographic ordering')

In [ ]:
if not _ace_descriptors_exist:
    ACE_file = os.path.join(descriptorslocation, f'{dataset}-ACE-CNAV.csv')
    
    CNAV_ACE.to_csv(ACE_file)

# Compare to DFT relaxed

In [ ]:
if not _ace_descriptors_exist:
    AtomsObjectsRLX = pd.read_pickle(os.path.join(dataset, 'Atomsobjects', f'{dataset}-POSCAR.relaxed-all-noscaled-AtomsObjects.pkl')).dropna()

In [ ]:
if not _ace_descriptors_exist:
    ACE_RLX = AtomsObjectsRLX['atoms'].map(ACEer.get_ace_projections)
    ACE_RLX.name = 'ace_projections'

In [ ]:
if not _ace_descriptors_exist:
    ACE_RLX

In [ ]:
if not _ace_descriptors_exist:
    expand_ace_rlx = gf.array_expansions(ACE_RLX.to_frame(),['ace_projections']) 

In [ ]:
if not _ace_descriptors_exist:
    CNAV_ACE_RLX = gf.featurize_dataframe(expand_ace_rlx.loc[intersection], CNList.loc[intersection]).astype(np.single)

In [ ]:
if not _ace_descriptors_exist:
    CNAV_ACE_RLX

In [ ]:
if not _ace_descriptors_exist:
    from sklearn.metrics import r2_score

In [ ]:
if not _ace_descriptors_exist:
    for i in range(6):
        fw, fh = plt.rcParams['figure.figsize']
        fig,axs = plt.subplots(1,5, figsize=(fw*5/2, fh))
        for CN, ax in zip(['0','CN12', 'CN14', 'CN15', 'CN16'], axs):
            feature = f'ace_projections_{i}_{CN}'
            x = CNAV_ACE_RLX[feature]
            y = CNAV_ACE[feature]
        #    reg = np.poly1d(np.polyfit(x, y, 1))
        #    ytilde = reg(x)
            r2 = r2_score(y, x)
            ax.scatter(x, y , ec='k')
            proj_0_range = [CNAV_ACE_RLX[feature].min(), CNAV_ACE_RLX[feature].max()] 
            ax.plot(proj_0_range, proj_0_range, 'k')
            ax.set_title(feature.replace('ace_projections', 'ACE')+f', $R^2$ = {r2:.2f}')
        axs[0].set_ylabel('guess structure')
        fig.supxlabel('DFT relaxed')
        fig.tight_layout()

In [ ]:
if not _ace_descriptors_exist:
    plt.rc('text', usetex=True)
    plt.rc('font', family='serif', size=24)

In [ ]:
if not _ace_descriptors_exist:
    X=    np.ravel(CNAV_ACE)
    Y=    np.ravel(CNAV_ACE_RLX)
    fig = plt.figure()
    ax = fig.add_subplot([0.25, 0.2, 0.65, 0.7])
    ax.scatter(X,Y, ec='k')
    #plt.yscale('log')
    #plt.xscale('log')
    therange = [X.min(), X.max()]
    r2 = r2_score(X,Y)
    plt.ylabel('ACE DFT relaxed')
    plt.xlabel('ACE guess structures')
    plt.plot(therange,therange, '-k')
    plt.title(f'ACE, $R^2 = ${r2:.5f}', fontsize=18)
    plt.savefig('Fe-Mo/graphs/Figure_Fe-Mo_ACE_rlx-vs-ini.png', dpi = 300)
    

# Prepare for prediction

## ACE no zero

In [ ]:
if not _ace_descriptors_exist:
    components

In [ ]:
if not _ace_descriptors_exist:
    nozero_acer = MyPyACECalculator(components, multispace_basis_config=AceConfig, exclude_ls=[0])

In [ ]:
if not _ace_descriptors_exist:
    all_funcspecs = []
    for block in nozero_acer.new_bbasis.funcspecs_blocks:
        [all_funcspecs.append(spec) for spec in block.funcspecs]
        

In [ ]:
if not _ace_descriptors_exist:
    all_funcspecs

In [ ]:
if not _ace_descriptors_exist:
    len(all_funcspecs)

In [ ]:
if not _ace_descriptors_exist:
    NOZEROACEFEATURES = AtomsObjects['atoms'].map(nozero_acer.get_ace_projections)
    NOZEROACEFEATURES.name = 'light_ace_projections'

In [ ]:
if not _ace_descriptors_exist:
    expand_NOZEROACEFEATURES = gf.array_expansions(NOZEROACEFEATURES.to_frame(), ['light_ace_projections'])

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOZEROACEFEATURES = gf.featurize_dataframe(expand_NOZEROACEFEATURES, CNList).astype(np.single)

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOZEROACEFEATURES.shape

In [ ]:
if not _ace_descriptors_exist:
    CNAV_ACE.shape

In [ ]:
if not _ace_descriptors_exist:
    NOZERO_ACE_file = os.path.join(descriptorslocation, f'{dataset}-NOZERO-ACE-CNAV.csv')
    
    CNAV_NOZEROACEFEATURES.to_csv(NOZERO_ACE_file)

## NOZERO NOONE ACE

In [ ]:
if not _ace_descriptors_exist:
    nozero_noone_acer = MyPyACECalculator(components, multispace_basis_config=AceConfig, exclude_ls=[0,1])

In [ ]:
if not _ace_descriptors_exist:
    all_funcspecs_nozeronoone = []
    for block in nozero_noone_acer.new_bbasis.funcspecs_blocks:
        [all_funcspecs_nozeronoone.append(spec) for spec in block.funcspecs]
        

In [ ]:
if not _ace_descriptors_exist:
    nozero_noone_acer.bbasis_configuration

In [ ]:
if not _ace_descriptors_exist:
    NOZERO_NOONE_ACEFEATURES = AtomsObjects['atoms'].map(nozero_noone_acer.get_ace_projections)
    NOZERO_NOONE_ACEFEATURES.name = 'light_ace_projections'

In [ ]:
if not _ace_descriptors_exist:
    expand_NOZERO_NOONE_ACEFEATURES = gf.array_expansions(NOZERO_NOONE_ACEFEATURES.to_frame(), ['light_ace_projections'])

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOZERO_NOONE_CEFEATURES = gf.featurize_dataframe(expand_NOZERO_NOONE_ACEFEATURES, CNList).astype(np.single)

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOZERO_NOONE_CEFEATURES.shape

In [ ]:
if not _ace_descriptors_exist:
    CNAV_ACE.shape

In [ ]:
if not _ace_descriptors_exist:
    NOZERO_NOONE_ACE_file = os.path.join(descriptorslocation, f'{dataset}-NOZERO_NOONE-ACE-CNAV.csv')
    
    CNAV_NOZERO_NOONE_CEFEATURES.to_csv(NOZERO_NOONE_ACE_file)

## NOZERO NOONE NOTWO ACE

In [ ]:
if not _ace_descriptors_exist:
    nozero_noone_notwo_acer = MyPyACECalculator(components, multispace_basis_config=AceConfig, exclude_ls=[0,1,2])

In [ ]:
if not _ace_descriptors_exist:
    all_funcspecs_nozeronoonenotwo = []
    for block in nozero_noone_notwo_acer.new_bbasis.funcspecs_blocks:
        [all_funcspecs_nozeronoonenotwo.append(spec) for spec in block.funcspecs]
        

In [ ]:
if not _ace_descriptors_exist:
    nozero_noone_notwo_acer.bbasis_configuration

In [ ]:
if not _ace_descriptors_exist:
    NOZERO_NOONE_NOTWO_ACEFEATURES = AtomsObjects['atoms'].map(nozero_noone_notwo_acer.get_ace_projections)
    NOZERO_NOONE_NOTWO_ACEFEATURES.name = 'light_ace_projections'

In [ ]:
if not _ace_descriptors_exist:
    expand_NOZERO_NOONE_NOTWO_ACEFEATURES = gf.array_expansions(NOZERO_NOONE_NOTWO_ACEFEATURES.to_frame(), ['light_ace_projections'])

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOZERO_NOONE_NOTWO_CEFEATURES = gf.featurize_dataframe(expand_NOZERO_NOONE_NOTWO_ACEFEATURES, CNList).astype(np.single)

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOZERO_NOONE_NOTWO_CEFEATURES.shape

In [ ]:
if not _ace_descriptors_exist:
    CNAV_ACE.shape

In [ ]:
if not _ace_descriptors_exist:
    NOZERO_NOONE_NOTWO_ACE_file = os.path.join(descriptorslocation, f'{dataset}-NOZERO_NOONE_NOTWO-ACE-CNAV.csv')
    
    CNAV_NOZERO_NOONE_CEFEATURES.to_csv(NOZERO_NOONE_NOTWO_ACE_file)

## NOTHREE ACE

In [ ]:
if not _ace_descriptors_exist:
    nothree_acer = MyPyACECalculator(components, multispace_basis_config=AceConfig, exclude_ls=[3])

In [ ]:
if not _ace_descriptors_exist:
    all_funcspecs_nothree = []
    for block in nothree_acer.new_bbasis.funcspecs_blocks:
        [all_funcspecs_nothree.append(spec) for spec in block.funcspecs]
        

In [ ]:
if not _ace_descriptors_exist:
    NOTHREE_ACEFEATURES = AtomsObjects['atoms'].map(nothree_acer.get_ace_projections)
    NOTHREE_ACEFEATURES.name = 'light_ace_projections'

In [ ]:
if not _ace_descriptors_exist:
    expand_NOTHREE_ACEFEATURES = gf.array_expansions(NOTHREE_ACEFEATURES.to_frame(), ['light_ace_projections'])

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOTHREE_ACEFEATURES = gf.featurize_dataframe(expand_NOTHREE_ACEFEATURES, CNList).astype(np.single)

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOTHREE_ACEFEATURES.shape

In [ ]:
if not _ace_descriptors_exist:
    NOTHREE_ACE_file = os.path.join(descriptorslocation, f'{dataset}-NOTHREE-ACE-CNAV.csv')
    
    CNAV_NOTHREE_ACEFEATURES.to_csv(NOTHREE_ACE_file)

## NOTHREE NOTWO ACE

In [ ]:
if not _ace_descriptors_exist:
    nothree_notwo_acer = MyPyACECalculator(components, multispace_basis_config=AceConfig, exclude_ls=[2,3])

In [ ]:
if not _ace_descriptors_exist:
    all_funcspecs_nothree_notwo = []
    for block in nothree_notwo_acer.new_bbasis.funcspecs_blocks:
        [all_funcspecs_nothree_notwo.append(spec) for spec in block.funcspecs]

In [ ]:
if not _ace_descriptors_exist:
    NOTHREE_NOTWO_ACEFEATURES = AtomsObjects['atoms'].map(nothree_notwo_acer.get_ace_projections)
    NOTHREE_NOTWO_ACEFEATURES.name = 'light_ace_projections'

In [ ]:
if not _ace_descriptors_exist:
    expand_NOTHREE_NOTWO_ACEFEATURES = gf.array_expansions(NOTHREE_NOTWO_ACEFEATURES.to_frame(), ['light_ace_projections'])

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOTHREE_NOTWO_ACEFEATURES = gf.featurize_dataframe(expand_NOTHREE_NOTWO_ACEFEATURES, CNList).astype(np.single)

In [ ]:
if not _ace_descriptors_exist:
    NOTHREE_NOTWO_ACE_file = os.path.join(descriptorslocation, f'{dataset}-NOTHREE_NOTWO-ACE-CNAV.csv')
    
    CNAV_NOTHREE_NOTWO_ACEFEATURES.to_csv(NOTHREE_NOTWO_ACE_file)

## NOTHREE NOTWO NOONE ACE

In [ ]:
if not _ace_descriptors_exist:
    nothree_notwo_noone_acer = MyPyACECalculator(components, multispace_basis_config=AceConfig, exclude_ls=[1,2,3])

In [ ]:
if not _ace_descriptors_exist:
    all_funcspecs_nothree_notwo_noone = []
    for block in nothree_notwo_noone_acer.new_bbasis.funcspecs_blocks:
        [all_funcspecs_nothree_notwo_noone.append(spec) for spec in block.funcspecs]

In [ ]:
if not _ace_descriptors_exist:
    NOTHREE_NOTWO_NOONE_ACEFEATURES = AtomsObjects['atoms'].map(nothree_notwo_noone_acer.get_ace_projections)
    NOTHREE_NOTWO_NOONE_ACEFEATURES.name = 'light_ace_projections'

In [ ]:
if not _ace_descriptors_exist:
    expand_NOTHREE_NOTWO_NOONE_ACEFEATURES = gf.array_expansions(NOTHREE_NOTWO_NOONE_ACEFEATURES.to_frame(), ['light_ace_projections'])

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOTHREE_NOTWO_NOONE_ACEFEATURES = gf.featurize_dataframe(expand_NOTHREE_NOTWO_NOONE_ACEFEATURES, CNList).astype(np.single)

In [ ]:
if not _ace_descriptors_exist:
    CNAV_NOTHREE_NOTWO_NOONE_ACEFEATURES.shape

In [ ]:
if not _ace_descriptors_exist:
    CNAV_ACE.shape

In [ ]:
if not _ace_descriptors_exist:
    NOTHREE_NOTWO_NOONE_ACE_file = os.path.join(descriptorslocation, f'{dataset}-NOTHREE_NOTWO_NOONE-ACE-CNAV.csv')
    
    CNAV_NOTHREE_NOTWO_NOONE_ACEFEATURES.to_csv(NOTHREE_NOTWO_NOONE_ACE_file)
    (NOTHREE_NOTWO_NOONE_ACE_file)

## Canonical 

In [ ]:
if not _ace_descriptors_exist:
    CanonicalAceConfig = copy.copy(AceConfig)

In [ ]:
if not _ace_descriptors_exist:
    CanonicalAceConfig['elements'] = ['W']

In [ ]:
if not _ace_descriptors_exist:
    CanonicalACEer = MyPyACECalculator(components = CanonicalAceConfig['elements'], multispace_basis_config=CanonicalAceConfig)

In [ ]:
if not _ace_descriptors_exist:
    canonicalatomsobjects = AtomsObjects['atoms'].map(reset_symbols)

In [ ]:
if not _ace_descriptors_exist:
    canonicalacedescriptors = canonicalatomsobjects.map(CanonicalACEer.get_ace_projections)
    canonicalacedescriptors.name = 'canonical_ace_projections'

In [ ]:
if not _ace_descriptors_exist:
    expand_canonical_ace = gf.array_expansions(canonicalacedescriptors.to_frame(),['canonical_ace_projections']) 

In [ ]:
if not _ace_descriptors_exist:
    CNAV_CANONICAL_ACE = gf.featurize_dataframe(expand_canonical_ace, CNList).astype(np.single)

In [ ]:
if not _ace_descriptors_exist:
    CNAV_CANONICAL_ACE

In [ ]:
if not _ace_descriptors_exist:
    canonical_ACE_file = os.path.join(descriptorslocation, f'{dataset}-canonical-ACE-CNAV.csv')

In [ ]:
if not _ace_descriptors_exist:
    CNAV_CANONICAL_ACE.to_csv(canonical_ACE_file)

# I think all the following happened in a conversation with Yuri:

``` python
new_multispace_config = copy.deepcopy(Test.default_multispace_config)
new_multispace_config['embeddings'] = {                                                                                  
     'Cr': {
         'npot': 'FinnisSinclairShiftedScaled',
         'fs_parameters': [1, 1], ## non-linear embedding function: 1*rho_1^1 + 1*rho_2^0.5
         'ndensity': 1,
         'rho_core_cut': 2000,
         'drho_core_cut': 250
         }, 

     'Co': {
         'npot': 'FinnisSinclairShiftedScaled', ## linear embedding function: 1*rho_1^1
         'fs_parameters': [1, 1],
         'ndensity': 1,
         'rho_core_cut': 3000,
         'drho_core_cut': 150
         }, 
    'W': {
         'npot': 'FinnisSinclairShiftedScaled', ## linear embedding function: 1*rho_1^1
         'fs_parameters': [1, 1],
         'ndensity': 1,
         'rho_core_cut': 3000,
         'drho_core_cut': 150
         }
     }
new_multispace_config['functions'] = {
                     'UNARY':  {
                             'nradmax_by_orders': [5, 1, 1, 1, 1],
                             'lmax_by_orders': [ 0, 3, 2,1,0],
                             },
#                      'Mo':  {
#                              'nradmax_by_orders': [5, 1, 1, 1, 1],
#                              'lmax_by_orders': [ 0, 3, 2,1,0],
#                              },
                     'BINARY':  {
                             'nradmax_by_orders': [5, 1, 1, 1, 1],
                             'lmax_by_orders': [ 0, 3, 2,1,0],
                             }
    }

# new_multispace_config['functions'] = {
#                      'ALL':  {
#                              'nradmax_by_orders': [5, 1, 1, 1, 1],
#                              'lmax_by_orders': [ 0, 3, 2,1,0],
#                              }
#     }

bas= construct_bbasisconfiguration(new_multispace_config)

bas.funcspecs_blocks[2].funcspecs

bas.funcspecs_blocks[2].funcspecs

construct_bbasisconfiguration??

Test.default_multispace_config

construct_bbasisconfiguration(Test.default_multispace_config)



ShorterFunctionsOptions = copy.deepcopy(default_options)

ShorterFunctionsOptions['functions'] = new_functions

ShorterFunctionsACE = MyPyACECalculator(components=['Fe','Mo'], multispace_basis_config=ShorterFunctionsOptions)

ACEFEATURES_shorter_functions = AtomsObjects['atoms'].map(ShorterFunctionsACE.get_ace_projections)
ACEFEATURES.name = 'ace_projections_shorter_functions'
```

In [ ]:
if not _ace_descriptors_exist:
    
    pass

In [ ]:
if not _ace_descriptors_exist:
    
    pass